# 🔍 Phase 4 — Inferensi OCR Plat Nomor (ALPR) — **Uji 4**

**Proyek:** PMLD — Indonesian License Plate Recognition
**PRD:** `PRD_decision_plat.md` (D3/D4/D7/D8) + **`PRD_decision_plat_v2.md` (V1–V7: Locked-Plate Display)**
**Uji ke-4:** `sample_video_1min_fix.mp4` — tampilan plat terkunci per track (anti flip-flop)

| Fitur | Status | Keterangan |
| :-- | :-- | :-- |
| D3 Character-position correction | ✅ | O↔0, I↔1, Z↔2, S↔5, B↔8, G↔6 sesuai template posisi |
| D4 Temporal voting | ✅ | Track kendaraan via IoU + majority vote |
| **V1 Locked-Plate Display** | ✅ **BARU** | Teks tampilan = hasil voting track, **bukan** bacaan per-frame |
| **V2 Lock rule** | ✅ **BARU** | Teks dikunci setelah `vote_count ≥ LOCK_MIN` |
| **V3 Override rule** | ✅ **BARU** | Kunci berubah hanya jika kandidat baru ≥ 2× vote + conf lebih tinggi |
| V4 Schema JSONL v2.1 | ✅ **BARU** | Field `plat_display`, `display_locked` |
| D8 Regex per-domain | ✅ | Plat Indonesia & UK dievaluasi terpisah |

**Perubahan kunci uji 3 → uji 4:** sebelumnya `annotate()` menampilkan `plat_corrected` per-frame
(hasil voting hanya dipakai di rekap) → teks berubah-ubah walau OCR sudah benar. Sekarang tampilan
memakai `plat_display` dari `tracker.best(track_id)` — **terkunci setelah cukup bukti**.

**Model:** `best_ft.pt` (fine-tune) jika ada, fallback `best.pt` (r3 baseline).
**OCR:** EasyOCR `english_g2` (bawaan) atau custom `easyocr_ft` jika ada.

> **Indikator visual:** hijau `▮` = LOCKED (tidak akan berubah) · oranye `…` = belum stabil · merah = tidak valid.

## 1️⃣ Cek GPU

EasyOCR jauh lebih cepat di GPU (CPU masih bisa, tapi video akan lambat).

In [ ]:
import torch

print('CUDA tersedia :', torch.cuda.is_available())
print('GPU           :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — video akan lambat, pertimbangkan T4')
# Tidak di-assert: inferensi citra tetap bisa jalan di CPU.

## 2️⃣ Mount Google Drive & Path Proyek

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/PMLD_FT'
MODEL_PATH  = f'{PROJECT_DIR}/model/best_ft.pt' if os.path.exists(f'{PROJECT_DIR}/model/best_ft.pt') else f'{PROJECT_DIR}/model/best.pt'
OCR_CUSTOM  = f'{PROJECT_DIR}/model/easyocr_ft' if os.path.isdir(f'{PROJECT_DIR}/model/easyocr_ft') else None
DATASET_DIR = f'{PROJECT_DIR}/dataset'
VIDEO_PATH  = f'{PROJECT_DIR}/test_video/demoparking.mp4'  # video uji domain target; ganti sample_vidio_1.mp4 untuk kontrol regresi
VIDEO_TAG   = 'demoparking.mp4'

# ---- Output: 1 folder utama per uji (PRD 01 §4) ----
RUN_NAME  = f'uji_8_{VIDEO_TAG}'  # workspace baru mulai dari uji_8 (uji 1–7c = arsip di PMLD lama)
RUN_DIR   = f'{PROJECT_DIR}/output/{RUN_NAME}'
OUT_IMG      = f'{RUN_DIR}/annotated_images'
OUT_TXT      = f'{RUN_DIR}/predictions_txt'
OUT_VIDEO    = f'{RUN_DIR}/processed_video'
OUT_CAPTURES = f'{RUN_DIR}/video_captures'
for d in (OUT_IMG, OUT_TXT, OUT_VIDEO, OUT_CAPTURES):
    os.makedirs(d, exist_ok=True)

# Guard
assert os.path.exists(MODEL_PATH), f'Model tidak ditemukan: {MODEL_PATH}'
assert os.path.isdir(f'{DATASET_DIR}/images/test'), 'Split test tidak ditemukan.'
assert os.path.exists(VIDEO_PATH), f'Video uji tidak ditemukan: {VIDEO_PATH}'

print(f'✅ Model: {MODEL_PATH}')
print(f'✅ OCR custom: {OCR_CUSTOM or "bawaan (english_g2)"}')
print(f'✅ Video: {VIDEO_PATH}')
print(f'📁 Output → {RUN_DIR}/')

## 3️⃣ Konfigurasi Inferensi

Semua parameter dalam satu sel — ubah di sini saja.

In [ ]:
# ---- Deteksi ----
CONF_DET     = 0.35     # Ambang batas keyakinan minimum agar plat terdeteksi oleh YOLO
IMG_SIZE     = 640       # Hemat VRAM (PRD migrasi FR-6); naikkan ke 1024 bila plat kecil/jauh kurang terbaca
PADDING      = 0.12     # Persentase area tambahan di sekitar bbox agar karakter pinggir plat tidak terpotong
MIN_HEIGHT   = 25       # Tinggi minimum bbox (piksel) untuk diproses, menapis plat yang terlalu kecil/jauh

# ---- OCR ----
OCR_LANGS    = ['en']   # Bahasa EasyOCR, menggunakan 'en' untuk abjad/angka Latin tanpa bias kata
ALLOWLIST    = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789' # Daftar karakter yang diizinkan (mengabaikan simbol/noise)
MIN_OCR_CONF = 0.20     # Ambang batas keyakinan minimum pembacaan teks oleh EasyOCR
UPSCALE      = 4        # Pengali perbesaran gambar crop plat sebelum dibaca EasyOCR agar lebih jelas

# ---- Video ----
FRAME_SKIP   = 5        # Frame skipping — hemat VRAM (PRD migrasi FR-6); 1 = proses semua frame
CAPTURE_EVERY_S = 10     # Interval waktu (detik) untuk menyimpan tangkapan layar frame ke folder output
N_SAMPLE_IMG = 8        # Jumlah maksimal sampel gambar ber-annotasi yang diambil untuk preview
SEED         = 42       # Seed acak agar pemilihan sampel gambar konsisten tiap kali dijalankan

# ---- D3: Character-position correction ----
CHAR_FIX     = True     # Mengaktifkan koreksi karakter berdasarkan tata letak format plat (misal: O/0, I/1)

# ---- D4: Temporal voting ----
TEMPORAL      = True    # Mengaktifkan sistem penjejakan (tracking) & akumulasi voting teks antar-frame
VOTE_MIN      = 3       # Jumlah vote minimum agar teks dianggap valid secara umum
IOU_TRACK     = 0.30    # Ambang batas tumpang-tindih bbox antar-frame untuk menghubungkan ID kendaraan yang sama
MAX_FRAME_GAP = 15      # Toleransi jumlah frame terputus sebelum sistem membuat Track ID baru

# ---- PRD v2: Locked-Plate Display ----
DISPLAY_LOCK   = True   # Menampilkan teks hasil voting pada video (bukan bacaan mentah per-frame)
LOCK_MIN       = 5      # Jumlah vote minimum sebelum tampilan teks plat dikunci secara permanen
OVERRIDE_RATIO = 2.0    # Rasio vote kandidat baru (2x) yang dibutuhkan untuk mengganti teks ter-lock

# ---- Line Crossing Detection (BARU — PRD migrasi FR-7) ----
LINE_CROSSING    = True       # True = deteksi penuh hanya saat objek melewati garis aktivasi
LC_ORIENTATION   = 'vertical' # 'vertical' (garis tegak) | 'horizontal' — demotol: kendaraan maju menjauh, plat bergerak turun
LC_ACTIVATE_AT   = 0.45       # posisi garis aktivasi (relatif 0–1 thd lebar/tinggi frame) — diukur dari log uji_8
LC_DEACTIVATE_AT = 0.70       # posisi garis berhenti (None = gerbang tetap terbuka setelah aktif) — 4x buka/tutup di demotol
LC_PROBE_EVERY   = 1          # saat gerbang nonaktif: deteksi sampel tiap N frame terproses

# ---- D8: Domain regex ----
DOMAIN_REGEX = 'UK'   # Aturan regex format wilayah/negara yang diterapkan (auto/ID/UK)

print(f'CONF_DET={CONF_DET}  UPSCALE={UPSCALE}x  FRAME_SKIP={FRAME_SKIP}')
print(f'CHAR_FIX={CHAR_FIX}  TEMPORAL={TEMPORAL}  VOTE_MIN={VOTE_MIN}')
print(f'DISPLAY_LOCK={DISPLAY_LOCK}  LOCK_MIN={LOCK_MIN}  OVERRIDE_RATIO={OVERRIDE_RATIO}')
print(f'DOMAIN_REGEX={DOMAIN_REGEX}')
print(f'LINE_CROSSING={LINE_CROSSING}  {LC_ORIENTATION} @ {LC_ACTIVATE_AT} → {LC_DEACTIVATE_AT}')



## 4️⃣ Install Library

`ultralytics` untuk deteksi, `easyocr` untuk OCR. Unduhan pertama EasyOCR (~100 MB model deteksi+rekognisi teks) terjadi saat inisialisasi di sel berikutnya.

In [ ]:
!pip install -q ultralytics easyocr

import ultralytics, easyocr
from ultralytics import YOLO
import cv2
import numpy as np
print('ultralytics:', ultralytics.__version__, '| easyocr:', easyocr.__version__, '| opencv:', cv2.__version__)

## 5️⃣ Load Model & Siapkan Pipeline

Fungsi inti (satu blok, siap di-copy MLOps untuk REST API):
- `preprocess_plate()` — crop + grayscale + CLAHE (PRD 02 §1.1)
- `clean_plate_text()` — post-processing regex format plat Indonesia: `1-2 huruf + 1-4 angka + 0-3 huruf` (contoh: `B 1234 XY`)
- `detect_and_read()` — deteksi → crop → OCR → kembalikan list deteksi

In [ ]:
import re, json, time
from collections import Counter

model = YOLO(MODEL_PATH)
reader = easyocr.Reader(OCR_LANGS, gpu=torch.cuda.is_available())
print('✅ Model & EasyOCR siap.')

# ---------- D3: Character-position correction ----------
LETTER_TO_DIGIT = {'O':'0','I':'1','L':'1','Z':'2','S':'5','G':'6','B':'8','A':'4','T':'7','E':'3','Q':'0','D':'0'}
DIGIT_TO_LETTER = {'0':'O','1':'I','2':'Z','5':'S','6':'G','8':'B'}
_FORBID_SUBS = {'Q','W','V'}

TEMPLATES = {
    'ID': [
        ('LDDDDLLL','ID H1A4H3'),('LDDDDLL','ID H1A4H2'),('LDDDDL','ID H1A4H1'),('LDDDD','ID H1A4'),
        ('LDDDLLL','ID H1A3H3'),('LDDDLL','ID H1A3H2'),('LDDDL','ID H1A3H1'),('LDDD','ID H1A3'),
        ('LDDLLL','ID H1A2H3'),('LDDLL','ID H1A2H2'),('LDDL','ID H1A2H1'),('LDD','ID H1A2'),
        ('LDLLL','ID H1A1H3'),('LDLL','ID H1A1H2'),('LDL','ID H1A1H1'),
        ('LLDDDDLLL','ID H2A4H3'),('LLDDDDLL','ID H2A4H2'),('LLDDDDL','ID H2A4H1'),('LLDDDD','ID H2A4'),
        ('LLDDDLLL','ID H2A3H3'),('LLDDDLL','ID H2A3H2'),('LLDDDL','ID H2A3H1'),('LLDDD','ID H2A3'),
        ('LLDDLLL','ID H2A2H3'),('LLDDLL','ID H2A2H2'),('LLDDL','ID H2A2H1'),('LLDD','ID H2A2'),
        ('LLDLLL','ID H2A1H3'),('LLDLL','ID H2A1H2'),('LLDL','ID H2A1H1'),
    ],
    'UK': [('LLDDLLL','UK LLDDLLL')],
}

def _apply_template(text, template):
    out, cost = [], 0
    for ch, expect in zip(text, template):
        orig = ch
        if expect == 'L' and ch.isdigit():
            ch = DIGIT_TO_LETTER.get(ch, ch); cost += 1
        elif expect == 'D' and ch.isalpha():
            ch = LETTER_TO_DIGIT.get(ch, ch); cost += 1
        if ch != orig and ch in _FORBID_SUBS:
            cost += 2
        out.append(ch)
    return ''.join(out), cost

def correct_plate_text(raw, domain='auto'):
    t = re.sub(r'[^A-Z0-9]', '', str(raw).upper())
    if len(t) < 3:
        return t, '-', '-'
    candidates = []
    domains = [domain] if domain != 'auto' else ['UK', 'ID']
    for dom in domains:
        for tpl, label in TEMPLATES.get(dom, []):
            if len(tpl) != len(t):
                continue
            corrected, cost = _apply_template(t, tpl)
            dom_bias = 0 if (dom == 'UK' and len(t) == 7) else (1 if dom == 'ID' else 2)
            candidates.append((corrected, label, dom, cost, dom_bias))
    if not candidates:
        return t, '-', '-'
    def _sort_key(c):
        eff = c[3] - 1.5 if (c[2] == 'UK' and c[1] == 'UK LLDDLLL') else c[3]
        return (eff, c[4], c[1])
    best = min(candidates, key=_sort_key)
    corrected, label, dom = best[0], best[1], best[2]
    valid, _ = validate_plate(corrected, domain=dom)
    if not valid:
        for cand in sorted(candidates, key=_sort_key)[1:]:
            v2, _ = validate_plate(cand[0], domain=cand[2])
            if v2:
                return cand[0], cand[1], cand[2]
        return t, '-', '-'
    return corrected, label, dom

# ---------- D8: Regex per-domain ----------
PLATE_RE_ID = re.compile(r'^([A-Z]{1,2})\s?(\d{1,4})\s?([A-Z]{0,3})$')
PLATE_RE_UK = re.compile(r'^([A-Z]{2})\s?(\d{2})\s?([A-Z]{3})$')

def validate_plate(text, domain='auto'):
    t = re.sub(r'[^A-Z0-9 ]', '', str(text).upper()).strip()
    compact = t.replace(' ', '')
    if domain in ('auto', 'ID'):
        m = PLATE_RE_ID.match(compact)
        if m:
            return True, 'ID'
    if domain in ('auto', 'UK'):
        m = PLATE_RE_UK.match(compact)
        if m:
            return True, 'UK'
    return False, '-'

def format_plate(text, domain):
    t = re.sub(r'[^A-Z0-9]', '', str(text).upper())
    if domain == 'UK' and len(t) == 7:
        return f'{t[:2]} {t[2:4]} {t[4:]}'
    if domain == 'ID':
        m = PLATE_RE_ID.match(t)
        if m:
            w, n, s = m.groups()
            return f'{w} {n}' + (f' {s}' if s else '')
    return t

# ---------- D4 + PRD v2: Temporal voting ----------
class PlateTracker:
    def __init__(self, iou_thresh=0.25, max_gap=15, vote_min=3, min_conf=0.30,
                 lock_min=3, override_ratio=2.0):
        self.iou_thresh = iou_thresh
        self.max_gap = max_gap
        self.vote_min = vote_min
        self.min_conf = min_conf
        self.lock_min = lock_min
        self.override_ratio = override_ratio
        self.tracks = []
        self._next_id = 0
        self.override_events = []

    def _get(self, track_id):
        for tr in self.tracks:
            if tr['track_id'] == track_id:
                return tr
        return None

    @staticmethod
    def _avg_conf(tr, plate):
        vals = [v[1] for v in tr['votes'] if v[0] == plate]
        return sum(vals) / len(vals) if vals else 0.0

    def _iou(self, a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
        inter = iw * ih
        if inter <= 0:
            return 0.0
        area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
        area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
        union = area_a + area_b - inter
        return inter / union if union > 0 else 0.0

    def best(self, track_id):
        tr = self._get(track_id)
        if tr is None or not tr['votes']:
            return '', False
        counter = Counter(v[0] for v in tr['votes'])
        top_plate, top_votes = counter.most_common(1)[0]
        if tr['locked_plate']:
            locked = tr['locked_plate']
            locked_votes = counter.get(locked, 0)
            if (top_plate != locked
                    and top_votes >= self.override_ratio * max(locked_votes, 1)
                    and self._avg_conf(tr, top_plate) > self._avg_conf(tr, locked)):
                self.override_events.append({'track_id': track_id, 'frame': tr['last_frame'],
                                             'from': locked, 'to': top_plate,
                                             'votes_from': locked_votes, 'votes_to': top_votes})
                tr['locked_plate'] = top_plate
                return top_plate, True
            return locked, True
        if top_votes >= self.lock_min:
            tr['locked_plate'] = top_plate
            return top_plate, True
        return top_plate, False

    def update(self, frame_idx, detections):
        active = [t for t in self.tracks if frame_idx - t['last_frame'] <= self.max_gap]
        for det in detections:
            best, best_iou = None, 0.0
            for tr in active:
                score = self._iou(det['bbox'], tr['last_box'])
                if score > best_iou:
                    best, best_iou = tr, score
            if best is not None and best_iou >= self.iou_thresh:
                best['last_box'] = det['bbox']
                best['last_frame'] = frame_idx
                if det.get('plat_corrected') and det.get('ocr_conf', 0) >= self.min_conf:
                    best['votes'].append((det['plat_corrected'], det['ocr_conf'], frame_idx))
                det['track_id'] = best['track_id']
            else:
                tr = {'track_id': self._next_id, 'last_box': det['bbox'], 'last_frame': frame_idx,
                      'votes': [], 'locked_plate': None}
                self._next_id += 1
                if det.get('plat_corrected') and det.get('ocr_conf', 0) >= self.min_conf:
                    tr['votes'].append((det['plat_corrected'], det['ocr_conf'], frame_idx))
                self.tracks.append(tr)
                det['track_id'] = tr['track_id']
            det['vote_count'] = len([v for v in (best or self.tracks[-1])['votes']
                                     if v[0] == det.get('plat_corrected', '')])
            det['is_stable'] = det['vote_count'] >= self.vote_min
            if DISPLAY_LOCK:
                display, locked = self.best(det['track_id'])
                det['plat_display'] = display if display else det.get('plat_corrected', '')
                det['display_locked'] = locked
            else:
                det['plat_display'] = det.get('plat_corrected', '')
                det['display_locked'] = False
        return detections

    def get_final_plates(self):
        results = []
        for tr in self.tracks:
            if not tr['votes']:
                continue
            counter = Counter(v[0] for v in tr['votes'])
            best_plate, best_count = counter.most_common(1)[0]
            avg_conf = sum(v[1] for v in tr['votes'] if v[0] == best_plate) / best_count
            results.append({
                'track_id': tr['track_id'],
                'plat_final': tr['locked_plate'] or best_plate,
                'locked': bool(tr['locked_plate']),
                'vote_count': best_count,
                'total_votes': len(tr['votes']),
                'stability': round(best_count / len(tr['votes']), 4),
                'avg_ocr_conf': round(avg_conf, 4),
            })
        return sorted(results, key=lambda x: -x['vote_count'])

# ---------- Helper: Annotate Frame ----------
def annotate(img, detections):
    annotated = img.copy()
    for d in detections:
        x1, y1, x2, y2 = d['bbox']
        text = d.get('plat_display', d.get('plat_corrected', ''))
        is_locked = d.get('display_locked', False)

        # Tentukan warna bbox & teks
        if is_locked:
            color = (0, 255, 0) # Hijau jika terkunci
            suffix = " [LOCKED]"
        elif text:
            color = (0, 165, 255) # Oranye jika belum stabil
            suffix = " ..."
        else:
            color = (0, 0, 255) # Merah jika tidak terbaca
            suffix = "?"

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 3)
        if text:
            cv2.putText(annotated, f"{text}{suffix}", (x1, max(y1-10, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    return annotated

# ---------- Pipeline deteksi + OCR + D3 dengan gerbang bypass ----------
def preprocess_plate(img_bgr, box, pad=PADDING, upscale=UPSCALE):
    h, w = img_bgr.shape[:2]
    x1, y1, x2, y2 = box
    px, py = int((x2 - x1) * pad), int((y2 - y1) * pad)
    x1, y1 = max(0, x1 - px), max(0, y1 - py)
    x2, y2 = min(w, x2 + px), min(h, y2 + py)
    crop = img_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enh = clahe.apply(gray)
    if upscale and upscale > 1:
        enh = cv2.resize(enh, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)
    return enh

def detect_and_read(img_bgr, conf=CONF_DET, tracker=None, frame_idx=0, force_ocr=True):
    res = model.predict(img_bgr, conf=conf, imgsz=IMG_SIZE, verbose=False)[0]
    out = []
    for box, bconf in zip(res.boxes.xyxy.tolist(), res.boxes.conf.tolist()):
        box = [int(v) for v in box]
        small = (box[3] - box[1]) < MIN_HEIGHT

        plate_img = None if (small or not force_ocr) else preprocess_plate(img_bgr, box)
        text, oconf, valid = '', 0.0, False
        corrected, template, domain = '', '-', '-'

        if plate_img is not None:
            ocr = reader.readtext(plate_img, allowlist=ALLOWLIST)
            if ocr:
                raw = ' '.join(t[1] for t in ocr)
                oconf = float(min(t[2] for t in ocr))
                text = raw.strip().upper()
                if CHAR_FIX:
                    corrected, template, domain = correct_plate_text(raw, domain=DOMAIN_REGEX)
                else:
                    corrected = text
                valid, valid_domain = validate_plate(corrected, domain=domain if domain != '-' else 'auto')
                if valid:
                    domain = valid_domain
        det = {
            'bbox': box, 'det_conf': round(float(bconf), 4),
            'plat': text, 'plat_corrected': corrected, 'plat_valid': valid,
            'domain': domain, 'template': template,
            'ocr_conf': round(oconf, 4), 'skipped_small': bool(small)
        }
        if tracker is not None:
            det['frame'] = frame_idx
        out.append(det)
    if tracker is not None:
        tracker.update(frame_idx, out)
    return out

print('✅ Pipeline D3+D4+D8 + Gate-Bypass CPU siap.')

## 6️⃣ Uji Citra Test → `annotated_images/` + `predictions_txt/`

Menguji `N_SAMPLE_IMG` citra acak dari split **test** (PRD 02 §5 Langkah 4), menyimpan gambar beranotasi dan **log JSONL** (1 baris per deteksi: file, bbox, teks plat, confidence deteksi & OCR — format yang nanti dikonsumsi webhook Backend `/detections`).

In [ ]:
import random, glob
from IPython.display import Image as IPImage, display

random.seed(SEED)
test_imgs = sorted(glob.glob(f'{DATASET_DIR}/images/test/*.*'))
samples = random.sample(test_imgs, min(N_SAMPLE_IMG, len(test_imgs)))
print(f'Menguji {len(samples)} dari {len(test_imgs)} citra test...')

log_path = f'{OUT_TXT}/test_images_log.jsonl'
rows, n_valid = [], 0
for p in samples:
    img = cv2.imread(p)
    dets = detect_and_read(img)
    for d in dets:
        rows.append({'file': os.path.basename(p), **d})
        n_valid += int(d['plat_valid'] and d['ocr_conf'] >= MIN_OCR_CONF)
    cv2.imwrite(f"{OUT_IMG}/{os.path.splitext(os.path.basename(p))[0]}_annotated.jpg", annotate(img, dets))

with open(log_path, 'w') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')

print(f'\n✅ {len(samples)} gambar dianotasi → {OUT_IMG}')
print(f'✅ {len(rows)} deteksi dicatat → {log_path}')
print(f'   Plat valid & OCR conf ≥ {MIN_OCR_CONF}: {n_valid}/{len(rows)}')
print('\n--- Contoh hasil ---')
for p in sorted(glob.glob(f'{OUT_IMG}/*_annotated.jpg'))[:4]:
    display(IPImage(filename=p, width=650))

## 7️⃣ Evaluasi OCR

Menampilkan tabel hasil baca vs *ground truth* (nama file dataset ini umumnya = nomor plat, mis. `B1234XY.jpg`) dan estimasi akurasi — target **PRD 01 §5: OCR Text Accuracy ≥ 80%**.

> ⚠️ Catatan metodologi: sebagian nama file test (mis. `train018_jpg.rf.*.jpg`) bukan nomor plat sebenarnya, jadi skor ini **indikatif, bukan final**. Pembacaan manual oleh manusia tetap disarankan untuk laporan.

In [ ]:
import pandas as pd

df = pd.read_json(log_path, lines=True)

def norm(s):
    return re.sub(r'[^A-Z0-9]', '', str(s).upper())

def cer(a, b):
    """Character Error Rate sederhana (Levenshtein distance / len(gt))."""
    a, b = norm(a), norm(b)
    if not b:
        return 1.0 if a else 0.0
    # DP edit distance
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            prev, dp[j] = dp[j], min(dp[j] + 1, dp[j-1] + 1, prev + (ca != cb))
    return dp[-1] / len(b)

if len(df):
    df['gt_norm'] = df['file'].map(lambda f: norm(os.path.splitext(os.path.splitext(f)[0])[0].split('_jpg')[0]))
    df['exact_match'] = df.apply(lambda r: norm(r['plat_corrected']) == r['gt_norm'] and len(r['gt_norm']) >= 4, axis=1)
    df['char_acc'] = df.apply(lambda r: 1 - cer(r['plat_corrected'], r['gt_norm']), axis=1)
    df['cer'] = df.apply(lambda r: cer(r['plat_corrected'], r['gt_norm']), axis=1)

    print('=== Evaluasi OCR (uji 3, dengan D3 koreksi) ===')
    print(df[['file','plat','plat_corrected','domain','plat_valid','det_conf','ocr_conf','exact_match','char_acc','cer']].to_string(index=False))

    print(f"\n--- Ringkasan ---")
    print(f"Exact match : {df['exact_match'].mean()*100:.1f}% ({int(df['exact_match'].sum())}/{len(df)})")
    print(f"Char acc avg: {df['char_acc'].mean()*100:.1f}%  (target ≥ 80%)")
    print(f"CER avg     : {df['cer'].mean()*100:.1f}%  (target ≤ 20%)")

    print(f"\n--- Per domain ---")
    for dom in df['domain'].unique():
        if dom == '-':
            continue
        sub = df[df['domain'] == dom]
        print(f"  {dom}: {len(sub)} deteksi, valid={sub['plat_valid'].sum()}, exact={sub['exact_match'].mean()*100:.1f}%")
else:
    print('Tidak ada deteksi — turunkan CONF_DET atau cek model.')

## 8️⃣ Uji Video 1 Menit → `processed_video/`

Pipeline video (PRD 02 §3.5):
1. Baca frame demi frame dengan OpenCV
2. Deteksi + OCR tiap **frame ke-`FRAME_SKIP`**; frame di antaranya memakai anotasi terakhir (efisiensi)
3. Gambar bbox + teks plat di atas frame
4. Simpan ke `sample_video_1min_annotated.mp4` + log JSONL per deteksi

In [ ]:
import time
import json
import os
import cv2

# Membuka video utama untuk pemrosesan teroptimasi sejak awal
cap = cv2.VideoCapture(VIDEO_PATH)
fps   = cap.get(cv2.CAP_PROP_FPS) or 25
w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
dur   = total / fps
print(f'Video: {w}x{h} @ {fps:.1f} fps, {total} frame (~{dur:.1f} detik)')

out_path  = f'{OUT_VIDEO}/{VIDEO_TAG}_annotated.mp4'
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
vlog_path = f'{OUT_TXT}/{VIDEO_TAG}_log.jsonl'

tracker = PlateTracker(iou_thresh=IOU_TRACK, max_gap=MAX_FRAME_GAP, vote_min=VOTE_MIN, min_conf=MIN_OCR_CONF,
                       lock_min=LOCK_MIN, override_ratio=OVERRIDE_RATIO) if TEMPORAL else None

# ---- Line Crossing gate (PRD migrasi FR-7) ----
lc_gate = None
if LINE_CROSSING:
    import sys
    SCRIPTS_DIR = f'{PROJECT_DIR}/scripts'
    if SCRIPTS_DIR not in sys.path:
        sys.path.append(SCRIPTS_DIR)
    from line_crossing import LineCrossingGate
    lc_gate = LineCrossingGate(orientation=LC_ORIENTATION,
                               activate_at=LC_ACTIVATE_AT,
                               deactivate_at=LC_DEACTIVATE_AT,
                               frame_size=(w, h))
    print(f'🚧 Line crossing AKTIF: garis {LC_ORIENTATION} @ {LC_ACTIVATE_AT}' +
          (f' → berhenti @ {LC_DEACTIVATE_AT}' if LC_DEACTIVATE_AT is not None else ' (tanpa garis berhenti)'))

t0, idx, last_dets, n_rows, n_caps = time.time(), 0, [], 0, 0
gate_active = False
last_cap_t = -CAPTURE_EVERY_S

with open(vlog_path, 'w') as vlog:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if idx % FRAME_SKIP == 0:
            # 1. Gunakan deteksi YOLO ringan tanpa OCR terlebih dahulu untuk update gerbang
            raw_dets = detect_and_read(frame, tracker=None, frame_idx=idx, force_ocr=False)

            # 2. Perbarui status gerbang berdasarkan koordinat objek
            if lc_gate is not None:
                centers = [((d['bbox'][0] + d['bbox'][2]) / 2, (d['bbox'][1] + d['bbox'][3]) / 2) for d in raw_dets]
                gate_active = lc_gate.update(centers)
            else:
                gate_active = True

            # 3. Jalankan OCR secara penuh hanya jika gerbang berstatus aktif
            if gate_active:
                last_dets = detect_and_read(frame, tracker=tracker, frame_idx=idx, force_ocr=True)
            else:
                # Di luar jangkauan gerbang aktif, kosongkan agar tidak membuang resource komputasi
                last_dets = []

            for d in last_dets:
                vlog.write(json.dumps({'frame': idx, 'timestamp_s': round(idx / fps, 2),
                                       'file': os.path.basename(VIDEO_PATH),
                                       'gate_active': gate_active, **d}) + '\n')
                n_rows += 1

        # 4. Gambar anotasi box & teks
        annotated = annotate(frame, last_dets)
        if lc_gate is not None:
            annotated = lc_gate.draw(annotated)

        # Render Indikator Status Gerbang pada Layar
        status_text = f"GATE: {'ACTIVE (OCR ON)' if gate_active else 'STANDBY (OCR OFF)'}"
        status_color = (0, 255, 0) if gate_active else (0, 0, 255)
        cv2.putText(annotated, status_text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 3)

        t_sec = idx / fps
        if t_sec - last_cap_t >= CAPTURE_EVERY_S:
            cap_path = f'{OUT_CAPTURES}/{VIDEO_TAG}_t{int(t_sec):03d}s_frame{idx}.jpg'
            cv2.imwrite(cap_path, annotated)
            n_caps += 1
            last_cap_t = t_sec

        writer.write(annotated)
        idx += 1
        if idx % 200 == 0:
            print(f'  ...Processed {idx}/{total} frames ({time.time()-t0:.0f}s)')

cap.release()
writer.release()
mins = (time.time() - t0) / 60

print(f'\n✅ Video utama berhasil diproses secara optimal → {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)')
print(f'✅ {n_rows} deteksi dicatat → {vlog_path}')
print(f'✅ {n_caps} capture frame → {OUT_CAPTURES}/')
print(f'   {total} frame diproses dalam {mins:.1f} menit.')
print('   TARGET PRD 01 §5: Video Testing 100% selesai tanpa error —', 'TERCAPAI ✅' if idx == total else f'GAGAL ❌ (berhenti di frame {idx}/{total})')
print(f'⚡ Terintegrasi dengan Sukses: Beban CPU berhasil ditekan berkat bypass OCR!')

# Laporan override
if tracker is not None and tracker.override_events:
    print(f'\n⚠️ {len(tracker.override_events)} override (kunci berubah — audit V3):')
    for ev in tracker.override_events:
        print(f"   track #{ev['track_id']} f{ev['frame']}: {ev['from']} → {ev['to']} (votes {ev['votes_from']}→{ev['votes_to']})")
elif tracker is not None:
    print('\n✅ Tidak ada override — semua kunci stabil sejak awal.')

# Statistik line crossing
if lc_gate is not None:
    st = lc_gate.stats()
    print(f"\n🚧 Line crossing: {st['orientation']} @ {st['activate_at']}"
          + (f" → {st['deactivate_at']}" if st['deactivate_at'] is not None else ' (tanpa garis berhenti)'))
    print(f"   gerbang dibuka {st['cross_in']}x, ditutup {st['cross_out']}x")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from line_crossing import LineCrossingGate

# Ambil satu frame dari video untuk test render
cap_test = cv2.VideoCapture(VIDEO_PATH)
ok, frame_test = cap_test.read()
cap_test.release()

if ok:
    h_t, w_t = frame_test.shape[:2]
    # Inisialisasi gerbang dengan settingan aktif
    test_gate = LineCrossingGate(orientation=LC_ORIENTATION,
                                 activate_at=LC_ACTIVATE_AT,
                                 deactivate_at=LC_DEACTIVATE_AT,
                                 frame_size=(w_t, h_t))

    # Gambar garis secara paksa
    drawn_frame = test_gate.draw(frame_test.copy())

    # Tampilkan di notebook menggunakan matplotlib
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(drawn_frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Test Render Garis: {LC_ORIENTATION} (START: {LC_ACTIVATE_AT}, STOP: {LC_DEACTIVATE_AT})")
    plt.axis('off')
    plt.show()
else:
    print("Gagal membaca video uji untuk test render.")

## 9️⃣ Rekap Hasil Video

Ringkasan plat yang paling sering terbaca selama video (berguna untuk demo dashboard & verifikasi konsistensi OCR antar-frame).

In [ ]:
# Membaca file log utama hasil pemrosesan video teroptimasi
vdf = pd.read_json(vlog_path, lines=True)
valid = vdf[(vdf['plat_valid']) & (vdf['ocr_conf'] >= MIN_OCR_CONF)]
print(f'Total deteksi: {len(vdf)} | valid: {len(valid)}')

print('\n--- Plat paling sering terbaca (raw OCR) ---')
print(valid['plat'].value_counts().head(10).to_string() if len(valid) else '(belum ada yang valid)')

print('\n--- Plat paling sering terbaca (setelah D3 koreksi) ---')
print(valid['plat_corrected'].value_counts().head(10).to_string() if len(valid) else '(belum ada yang valid)')

# ===== PRD v2 §6: metrik utama — flip-flop tampilan setelah lock =====
print('\n=== PRD v2: Metrik Locked-Plate Display ===')
if 'plat_display' in vdf.columns and 'track_id' in vdf.columns:
    flips_total, flips_after_lock, tracks_checked = 0, 0, 0
    for tid, grp in vdf[vdf['track_id'].notna()].groupby('track_id'):
        grp = grp.sort_values('frame')
        disp = grp['plat_display'].tolist()
        locked = grp['display_locked'].tolist()
        tracks_checked += 1
        # transisi tampilan keseluruhan
        changes = sum(1 for a, b in zip(disp, disp[1:]) if a != b)
        flips_total += changes
        # transisi SETELAH lock (target: 0)
        for i in range(1, len(disp)):
            if locked[i] and disp[i] != disp[i-1] and disp[i-1] != '':
                flips_after_lock += 1
    print(f'  Track diperiksa              : {tracks_checked}')
    print(f'  Transisi tampilan (semua)    : {flips_total}')
    print(f'  Flip-flop SETELAH lock       : {flips_after_lock}  (target: 0)')
    print('  Status                       :', 'TERCAPAI ✅' if flips_after_lock == 0 else 'GAGAL ❌')

    locked_rows = vdf[vdf.get('display_locked', False) == True]
    print(f'  Deteksi dengan tampilan lock : {len(locked_rows)}/{len(vdf)} ({len(locked_rows)/max(len(vdf),1)*100:.1f}%)')
else:
    print('  (kolom plat_display/track_id tidak ada — jalankan sel video dulu)')

if TEMPORAL and tracker is not None:
    print('\n--- D4+V1: Plat final per track (locked = teks tidak akan berubah) ---')
    final = tracker.get_final_plates()
    for r in final[:10]:
        stab = r['stability'] * 100
        tag = 'LOCKED' if r['locked'] else 'sementara'
        print(f"  Track #{r['track_id']:2d}: {r['plat_final']:12s} [{tag:9s}] (votes={r['vote_count']}/{r['total_votes']}, stab={stab:.0f}%, conf={r['avg_ocr_conf']:.2f})")
    print(f'\n  Total track: {len(final)}')
    print(f'  Track terkunci: {sum(1 for r in final if r["locked"])}')
    stable = [r for r in final if r['stability'] >= 0.8]
    print(f'  Track stabil (≥80%): {len(stable)}')

print('\n--- Per domain ---')
for dom in vdf['domain'].unique():
    if dom == '-':
        continue
    sub = vdf[vdf['domain'] == dom]
    print(f"  {dom}: {len(sub)} deteksi, valid={sub['plat_valid'].sum()} ({sub['plat_valid'].mean()*100:.1f}%%)")

## ✅ Ringkasan & Serah Terima

Setelah semua sel lolos, deliverable uji 4 (PRD v2) lengkap:

```
MyDrive/PMLD_FT/output/uji_4_sample_video_1min_fix.mp4/
├── annotated_images/                  ← citra test + bbox + teks
├── predictions_txt/
│   ├── test_images_log.jsonl          ← schema v2
│   └── sample_video_1min_fix.mp4_log.jsonl  ← schema v2.1: +plat_display, display_locked
├── processed_video/
│   └── sample_video_1min_fix.mp4_annotated.mp4  ← tampilan TERKUNCI (anti flip-flop)
└── video_captures/                    ← capture tiap 10 detik
```

**Schema JSONL v2.1** (backward compatible):
```json
{
  "frame": 300, "timestamp_s": 10.0, "file": "sample_video_1min_fix.mp4",
  "bbox": [1207, 785, 1290, 807], "det_conf": 0.81,
  "plat": "GXI50GJ", "plat_corrected": "GX15OGJ",
  "plat_valid": true, "domain": "UK", "template": "UK LLDDLLL",
  "ocr_conf": 0.72, "skipped_small": false,
  "track_id": 7, "vote_count": 5, "is_stable": true,
  "plat_display": "GX15OGJ", "display_locked": true
}
```

**Target PRD v2 §6:**

| Metrik | Target | Cara ukur |
| :-- | :-- | :-- |
| Flip-flop setelah lock | **0** | sel 9 (metrik otomatis) |
| Track terkunci = hasil voting akhir | 100% | sel 9 |
| Override tercatat (bila terjadi) | audit trail | sel 8 |
| CER / akurasi (sel 7) | tidak menurun vs uji 3 | sel 7 |
| Video 100% tanpa error | ✅ | sel 8 |

**Perilaku tampilan:**
- 🟩 `GX 15 OGJ |` — LOCKED: teks tidak akan berubah sampai track berakhir
- 🟧 `GX 15 OGJ ..` — kandidat sementara, menunggu vote cukup (`LOCK_MIN=3`)
- 🟥 `plat? (0.62)` — deteksi tanpa pembacaan valid

**Langkah selanjutnya:** update `PHASE4_SETUP_NOTES.md`, lalu handover hasil uji 4 ke tim.
Jalur PRD v1 (fine-tune detector/recognizer) tetap menunggu pelabelan manual (Fase A.2).